# Claude Agent SDK Exploration

Hands-on notebook for exploring the Claude Agent SDK primitives directly.
Use this alongside iteration work to experiment with patterns before integrating them into the server.

**Prerequisites:**
- `.env` file at project root with Foundry credentials
- `uv sync` (includes `claude-agent-sdk` and `jupyterlab`)

**Run with:** `uv run jupyter lab` from the project root

## Setup

Load environment variables and configure MLflow tracing.

In [1]:
from dotenv import load_dotenv

load_dotenv()  # loads .env from project root

True

In [2]:
import mlflow
import mlflow.anthropic  # noqa: F811

mlflow.set_tracking_uri("http://localhost:5000")  # MLflow in Docker Compose
mlflow.set_experiment("notebook-exploration")
mlflow.anthropic.autolog()  # type: ignore[attr-defined]

print(f"MLflow tracking: {mlflow.get_tracking_uri()}")

2026/04/05 09:27:08 INFO mlflow.tracking.fluent: Experiment with name 'notebook-exploration' does not exist. Creating a new experiment.


MLflow tracking: http://localhost:5000


## 1. The Basics: ClaudeSDKClient and ClaudeAgentOptions

The two core primitives:
- **`ClaudeAgentOptions`** — configuration: what model, what tools, what permissions
- **`ClaudeSDKClient`** — a session: connect, send a query, receive messages

The client is an async context manager. Each `async with` block is one isolated session.

In [3]:
from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient, ResultMessage

# Minimal options — no tools, just a direct LLM call
basic_options = ClaudeAgentOptions(
    model="claude-haiku-4-5",
    permission_mode="bypassPermissions",
    max_turns=1,
)

In [8]:
import pprint

async with ClaudeSDKClient(options=basic_options) as client:
    await client.query("What is the capital of France? Answer in one sentence.")
    async for message in client.receive_response():
        print(type(message).__name__)
        pprint.pp(message.__dict__)

SystemMessage
{'subtype': 'init',
 'data': {'type': 'system',
          'subtype': 'init',
          'cwd': '/home/lambdakris/source/scholarbot/notebooks',
          'session_id': 'db28fe75-f62b-4eb5-a9ad-b3196f40e2d4',
          'tools': ['Task',
                    'AskUserQuestion',
                    'Bash',
                    'CronCreate',
                    'CronDelete',
                    'CronList',
                    'Edit',
                    'EnterPlanMode',
                    'EnterWorktree',
                    'ExitPlanMode',
                    'ExitWorktree',
                    'Glob',
                    'Grep',
                    'LSP',
                    'NotebookEdit',
                    'Read',
                    'Skill',
                    'TaskOutput',
                    'TaskStop',
                    'TodoWrite',
                    'ToolSearch',
                    'WebFetch',
                    'WebSearch',
                    'Write',
        

Trace(trace_id=tr-6f67bc49c5eff25abc28c7f67ea26a27)

## 2. Inspecting Message Types

The agent yields several message types. Let's collect them all and see what we get.

In [9]:
messages = []

async with ClaudeSDKClient(options=basic_options) as client:
    await client.query("Explain recursion in one sentence.")
    async for message in client.receive_response():
        messages.append(message)

for i, msg in enumerate(messages):
    print(f"[{i}] {type(msg).__name__}")
    # Show available attributes
    attrs = {k: v for k, v in vars(msg).items() if not k.startswith('_')}
    for k, v in attrs.items():
        val = str(v)[:120]
        print(f"    {k}: {val}")
    print()

[0] SystemMessage
    subtype: init
    data: {'type': 'system', 'subtype': 'init', 'cwd': '/home/lambdakris/source/scholarbot/notebooks', 'session_id': '295fb577-c4c

[1] AssistantMessage
    content: [ThinkingBlock(thinking="The user is asking for a one-sentence explanation of recursion. This is a straightforward quest
    model: claude-haiku-4-5-20251001
    parent_tool_use_id: None
    error: None
    usage: {'input_tokens': 10, 'cache_creation_input_tokens': 2865, 'cache_read_input_tokens': 18825, 'cache_creation': {'ephemera
    message_id: msg_01HWaCNPaToxNKvrmMfaKbiV
    stop_reason: None
    session_id: 295fb577-c4c9-4de3-8947-e30aae64cf2f
    uuid: 571cd191-046a-459c-aa2d-da673209f121

[2] AssistantMessage
    content: [TextBlock(text='Recursion is a technique where a function solves a problem by calling itself with simpler inputs until 
    model: claude-haiku-4-5-20251001
    parent_tool_use_id: None
    error: None
    usage: {'input_tokens': 10, 'cache_creation_input_toke

Trace(trace_id=tr-3b4519a8e9069510f17aa8b4467974b8)

## 3. Adding Tools: WebSearch

The agent becomes agentic when it has tools. `WebSearch` and `WebFetch` are built-in.
With tools, the model decides whether and when to use them — we don't control the flow.

In [10]:
search_options = ClaudeAgentOptions(
    model="claude-haiku-4-5",
    permission_mode="bypassPermissions",
    allowed_tools=["WebSearch", "WebFetch"],
    system_prompt=(
        "You are a research assistant. "
        "Use web search to find current information. "
        "Cite your sources with URLs."
    ),
    max_turns=10,
)

In [15]:
messages = []

async with ClaudeSDKClient(options=search_options) as client:
    await client.query("What is the current population of Tokyo?")
    async for message in client.receive_response():
        print(type(message).__name__)
        pprint.pp(message.__dict__)

SystemMessage
{'subtype': 'init',
 'data': {'type': 'system',
          'subtype': 'init',
          'cwd': '/home/lambdakris/source/scholarbot/notebooks',
          'session_id': '7fb95a50-d05c-4007-8cb8-2c37131c816c',
          'tools': ['Task',
                    'AskUserQuestion',
                    'Bash',
                    'CronCreate',
                    'CronDelete',
                    'CronList',
                    'Edit',
                    'EnterPlanMode',
                    'EnterWorktree',
                    'ExitPlanMode',
                    'ExitWorktree',
                    'Glob',
                    'Grep',
                    'LSP',
                    'NotebookEdit',
                    'Read',
                    'Skill',
                    'TaskOutput',
                    'TaskStop',
                    'TodoWrite',
                    'ToolSearch',
                    'WebFetch',
                    'WebSearch',
                    'Write',
        

Trace(trace_id=tr-fb420f2c1212b3d140cc27c17b91f55d)

## 4. Understanding the Trace

With `mlflow.anthropic.autolog()` active, every session is traced.
Let's look at the traces we've generated.

In [12]:
# Search for recent traces in our experiment
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name("notebook-exploration")

if experiment:
    traces = client.search_traces(experiment_ids=[experiment.experiment_id], max_results=5)
    print(f"Found {len(traces)} traces\n")
    for trace in traces:
        print(f"Trace: {trace.info.request_id}")
        print(f"  Status: {trace.info.status}")
        print(f"  Timestamp: {trace.info.timestamp_ms}")
        print()

Found 5 traces

Trace: tr-7c7e1cd145296e25a39f8e1709a20fb3
  Status: TraceStatus.OK
  Timestamp: 1775403639523

Trace: tr-3b4519a8e9069510f17aa8b4467974b8
  Status: TraceStatus.OK
  Timestamp: 1775403531700

Trace: tr-6f67bc49c5eff25abc28c7f67ea26a27
  Status: TraceStatus.OK
  Timestamp: 1775403309581

Trace: tr-07628342b21f3010a199cd887eea7320
  Status: TraceStatus.OK
  Timestamp: 1775403203305

Trace: tr-52aa3045c0900185bda65ed446f3128e
  Status: TraceStatus.OK
  Timestamp: 1775403063203



/tmp/ipykernel_39988/494420616.py:6: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = client.search_traces(experiment_ids=[experiment.experiment_id], max_results=5)


Check the MLflow UI at http://localhost:5000 for a visual view of the traces,
including the full span tree showing each tool call and LLM interaction.

## 5. System Prompts and Behavior

The `system_prompt` shapes how the agent behaves. Experiment with different prompts
and observe how the agent's tool usage and answer quality changes.

In [ ]:
# Try your own system prompt and question here
custom_options = ClaudeAgentOptions(
    model="claude-haiku-4-5",
    permission_mode="bypassPermissions",
    allowed_tools=["WebSearch", "WebFetch"],
    system_prompt="YOUR PROMPT HERE",
    max_turns=10,
)

async with ClaudeSDKClient(options=custom_options) as client:
    await client.query("YOUR QUESTION HERE")
    async for message in client.receive_response():
        if isinstance(message, ResultMessage):
            print(message.result)

## 6. Scratch Space

Use the cells below for ad-hoc experimentation during iteration work.